# Notebook 04 — Baseline Evaluation

Compares three summarisation approaches on the AMI test set to establish whether LiveNote's incremental M2 pipeline outperforms simple baselines.

| Approach | Method |
|---|---|
| **Lead-3** | First 3 sentences of the dialogue — cheapest extractive baseline |
| **BART** | `facebook/bart-large-cnn` applied to the full dialogue (truncated to 1024 tokens) |
| **LiveNote M2** | Single-pass through our `prompt_builder → llm_client → trust_validator` pipeline |

Metrics: ROUGE-1, ROUGE-2, ROUGE-L (F1) against AMI gold abstractive summaries.

**Dataset:** `knkarthick/AMI` — text-level (dialogues + summaries, no audio needed here).

In [ ]:
import os
import re
import sys
import time
from pathlib import Path

# ── Repo root on sys.path so backend.app.* is importable ──────────────────────
PROJECT_ROOT = (
    Path.cwd().resolve().parent
    if Path.cwd().name == 'Notebooks'
    else Path.cwd().resolve()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# ── Feature flags (no audio needed for text-level evaluation) ─────────────────
os.environ.setdefault('DIARIZATION_ENABLED', 'false')
os.environ.setdefault('LLM_MODE', 'ollama')
os.environ.setdefault('OLLAMA_MODEL', 'llama3.1:8b')  # update after NB10 if desired
os.environ.setdefault('OLLAMA_BASE_URL', 'http://localhost:11434')
os.environ.setdefault('OLLAMA_TIMEOUT_SEC', '90')

import pandas as pd
from datasets import load_dataset
from rouge_score import rouge_scorer

print('Environment ready.')

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────────
SAMPLE_SIZE = 30        # number of AMI meetings to evaluate
RAND_SEED   = 42
BARTMAX_TOKENS = 1024  # BART context limit

ds = load_dataset('knkarthick/AMI', split='test')
print(f'AMI test set size: {len(ds)}')
print(f'Columns: {ds.column_names}')
sample = ds.shuffle(seed=RAND_SEED).select(range(min(SAMPLE_SIZE, len(ds))))
sample

In [ ]:
# ── ROUGE helper ───────────────────────────────────────────────────────────────
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)


def rouge_row(reference: str, prediction: str) -> dict:
    if not reference.strip() or not prediction.strip():
        return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}
    scores = scorer.score(reference.strip(), prediction.strip())
    return {
        'rouge1': round(scores['rouge1'].fmeasure, 4),
        'rouge2': round(scores['rouge2'].fmeasure, 4),
        'rougeL': round(scores['rougeL'].fmeasure, 4),
    }


def get_dialogue(row: dict) -> str:
    """Return the raw dialogue string — handles str or list."""
    for col in ('dialogue', 'transcript', 'text'):
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            return val.strip()
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str) and item.strip():
                    return item.strip()
    return ''


def get_summary(row: dict) -> str:
    """Return the gold summary string — handles str or list (multiple annotators)."""
    for col in ('summary', 'abstract', 'abstractive_summary'):
        val = row.get(col)
        if isinstance(val, str) and val.strip():
            return val.strip()
        if isinstance(val, list):
            for item in val:
                if isinstance(item, str) and item.strip():
                    return item.strip()  # take first annotator's summary
    return ''


print('ROUGE helpers defined.')

## Approach 1 — Lead-3 Baseline

In [ ]:
def lead_3(dialogue: str) -> str:
    """Return the first 3 non-empty sentences from the dialogue."""
    sentences = re.split(r'(?<=[.!?])\s+', dialogue.strip())
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]
    return ' '.join(sentences[:3])


lead3_rows = []
for row in sample:
    dialogue = get_dialogue(row)
    reference = get_summary(row)
    prediction = lead_3(dialogue)
    scores = rouge_row(reference, prediction)
    lead3_rows.append({'approach': 'Lead-3', 'prediction': prediction[:200], **scores})

lead3_df = pd.DataFrame(lead3_rows)
print('Lead-3 mean ROUGE:')
lead3_df[['rouge1', 'rouge2', 'rougeL']].mean().round(4)

## Approach 2 — BART Baseline

> Requires `transformers` and `torch`. First run will download `facebook/bart-large-cnn` (~1.6 GB).

In [ ]:
from transformers import pipeline as hf_pipeline

bart = hf_pipeline(
    'summarization',
    model='facebook/bart-large-cnn',
    device=-1,          # CPU; set to 0 for GPU
    truncation=True,
    max_length=200,
    min_length=40,
)
print('BART pipeline loaded.')

In [ ]:
def bart_summarize(dialogue: str) -> str:
    # Truncate to BART_MAX_TOKENS words (rough proxy for tokens)
    words = dialogue.split()
    truncated = ' '.join(words[:BARTMAX_TOKENS])
    result = bart(truncated)
    return result[0]['summary_text'] if result else ''


bart_rows = []
for i, row in enumerate(sample):
    dialogue = get_dialogue(row)
    reference = get_summary(row)
    prediction = bart_summarize(dialogue)
    scores = rouge_row(reference, prediction)
    bart_rows.append({'approach': 'BART', 'prediction': prediction[:200], **scores})
    if (i + 1) % 5 == 0:
        print(f'  {i + 1}/{len(sample)} done')

bart_df = pd.DataFrame(bart_rows)
print('\nBART mean ROUGE:')
bart_df[['rouge1', 'rouge2', 'rougeL']].mean().round(4)

## Approach 3 — LiveNote M2 (Single-Pass)

Converts each dialogue into utterance dicts (with synthetic timestamps), creates a `MemoryState`, and runs one M2 extraction pass to get a summary.

> Requires Ollama running locally with `llama3.1:8b` (or the model set in `OLLAMA_MODEL`).

In [ ]:
from backend.app.models.memory import MemoryState
from backend.app.module2.intelligence_extractor import IntelligenceExtractor

extractor = IntelligenceExtractor()


def dialogue_to_utterances(dialogue: str) -> list[dict]:
    """Convert raw AMI dialogue string to utterance dicts with synthetic timestamps."""
    utterances = []
    t = 0.0
    for line in dialogue.strip().split('\n'):
        line = line.strip()
        if not line:
            continue
        if ':' in line:
            speaker_raw, text = line.split(':', 1)
            speaker = re.sub(r'\s+', '_', speaker_raw.strip().upper())
        else:
            speaker, text = 'SPEAKER_01', line
        text = text.strip()
        if not text:
            continue
        words = text.split()
        duration = max(1.5, len(words) * 0.4)  # ~0.4 s/word
        utterances.append({
            'speaker': speaker,
            'text': text,
            'start_time': round(t, 2),
            'end_time': round(t + duration, 2),
            'word_count': len(words),
            'confidence': 0.92,
        })
        t += duration + 0.3
    return utterances


def livenote_summarize(dialogue: str, meeting_id: str) -> str:
    utterances = dialogue_to_utterances(dialogue)
    if not utterances:
        return ''
    # Extract unique speakers
    speakers = sorted({u['speaker'] for u in utterances})
    from datetime import date
    state = MemoryState(
        meeting_id=meeting_id,
        meeting_start_date=date.today().isoformat(),
        known_speakers=speakers,
    )
    # Load all utterances into the LLM buffer (single-pass)
    state.llm_transcript_buffer = utterances
    try:
        extraction = extractor.run(state)
        return extraction.summary or state.running_summary
    except Exception as exc:
        print(f'  [{meeting_id}] M2 error: {exc}')
        return ''


print('LiveNote M2 functions defined.')

In [ ]:
livenote_rows = []
for i, row in enumerate(sample):
    meeting_id = str(row.get('id', f'meeting_{i}'))
    dialogue = get_dialogue(row)
    reference = get_summary(row)
    t0 = time.perf_counter()
    prediction = livenote_summarize(dialogue, meeting_id)
    latency = round(time.perf_counter() - t0, 2)
    scores = rouge_row(reference, prediction)
    livenote_rows.append({'approach': 'LiveNote-M2', 'latency_sec': latency,
                          'prediction': prediction[:200], **scores})
    if (i + 1) % 5 == 0:
        print(f'  {i + 1}/{len(sample)} done')

livenote_df = pd.DataFrame(livenote_rows)
print('\nLiveNote M2 mean ROUGE:')
livenote_df[['rouge1', 'rouge2', 'rougeL']].mean().round(4)

## Results Comparison

In [ ]:
all_df = pd.concat([lead3_df, bart_df, livenote_df], ignore_index=True)

comparison = (
    all_df.groupby('approach')[['rouge1', 'rouge2', 'rougeL']]
    .mean()
    .round(4)
    .sort_values('rouge1', ascending=False)
)
print('=== ROUGE Comparison (mean F1) ===')
comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=False)
for ax, metric in zip(axes, ['rouge1', 'rouge2', 'rougeL']):
    vals = comparison[metric]
    bars = ax.bar(vals.index, vals.values, color=['#4C72B0', '#55A868', '#C44E52'])
    ax.set_title(metric.upper())
    ax.set_ylim(0, max(vals.values) * 1.25)
    ax.set_ylabel('F1')
    for bar, val in zip(bars, vals.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                f'{val:.3f}', ha='center', va='bottom', fontsize=9)
plt.suptitle('Baseline Evaluation — ROUGE F1 Comparison', fontsize=13)
plt.tight_layout()
plt.savefig('04_baseline_rouge.png', dpi=150, bbox_inches='tight')
plt.show()

## Conclusion

Record the winner and its margin over baselines here after running.

| Finding | Value |
|---|---|
| Best approach | *(fill after run)* |
| ROUGE-1 vs Lead-3 delta | *(fill)* |
| ROUGE-1 vs BART delta | *(fill)* |
| Notes | *(fill)* |